# 02 — Merton Jump-Diffusion

**What this notebook is:** a complete start-to-finish guide for Merton’s model (GBM + sudden jumps), plus an interactive Monte Carlo playground.

**Previous model:** [`01_gbm.ipynb`](01_gbm.ipynb)

**Your project data:** `../data/equity/prices_clean.csv`, `log_returns_*.csv`, `summary_stats.csv`


## 1. Model idea

Merton = GBM continuous moves **plus** rare jumps (news, crashes, earnings).

**Continuous (differential):**

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa)\, dt + \sigma\, dW_t + (e^J - 1)\, dN_t$$

**Simulation (not differential):**

$$S_{t+\Delta t} = S_t \exp\Big(\big(\mu - \lambda\kappa - \tfrac{1}{2}\sigma^2\big)\Delta t + \sigma\sqrt{\Delta t}\, Z + \sum_{i=1}^{N_{\Delta t}} J_i\Big)$$

with $Z\sim N(0,1)$, $N_{\Delta t}\sim\mathrm{Poisson}(\lambda\Delta t)$, $J_i\sim N(\mu_J,\sigma_J^2)$.

**Jump compensation** (keeps the mean drift equal to $\mu$ after adding jumps):

$$\kappa = E[e^J-1] = e^{\mu_J + \sigma_J^2/2} - 1$$

| Symbol | Meaning |
|--------|---------|
| $\mu,\sigma$ | same roles as in GBM (diffusion part) |
| $\lambda$ | expected **number of jumps per year** |
| $N_t$ | Poisson jump counter |
| $J$ | log jump size |
| $\mu_J,\sigma_J$ | mean and std of log jump size |
| $\kappa$ | $E[e^J-1]$, computed from $\mu_J,\sigma_J$ |


## 2. End-to-end workflow

| Step | What you do | Output |
|------|-------------|--------|
| **1. Collect prices** | Adjusted closes for ticker / regime | Price series |
| **2. Log returns** | $r_t=\ln(S_t/S_{t-1})$ | Return series |
| **3. Estimate diffusion part** | Roughly like GBM: $\mu$, $\sigma$ from “normal” days (or jointly with jumps) | $\hat\mu,\hat\sigma$ |
| **4. Estimate jump part** | Intensity $\lambda$ and jump-size law $(\mu_J,\sigma_J)$ | Jump parameters |
| **5. Compute $\kappa$** | $\kappa=\exp(\mu_J+\sigma_J^2/2)-1$ | One derived constant |
| **6. Set design** | $S_0$, $T$, $\Delta t$, number of paths | Simulation grid |
| **7. Simulate paths** | Each step: diffusion shock + Poisson number of jumps + jump sizes | Price cloud with kinks |
| **8. Use paths** | Fat-tail stats, option pricing, regime comparison | Research outputs |

Same regimes as GBM (crisis / normal / late / COVID) — jump intensity is often **higher** in crisis and COVID.


## 3. How to calculate parameters from historical data

There is no single industry-standard closed form. A clear **beginner workflow** is:

### A. Diffusion parameters (GBM-like baseline)

Using all daily returns (or returns with jumps removed — see B):

$$\hat\mu = \bar{r}\times 252,\qquad \hat\sigma = s_r\times\sqrt{252}$$

If you remove jump days first, $\hat\sigma$ better represents the **smooth** diffusion vol.

### B. Detect jump days (simple threshold method)

1. Compute daily returns $r_t$.
2. Estimate a robust scale, e.g. $\hat\sigma_{\text{day}} = \mathrm{std}(r)$.
3. Flag a day as a jump if $|r_t| > c\cdot\hat\sigma_{\text{day}}$ with a threshold such as $c=3$ or $c=4$.

Let $n_{\text{jumps}}$ be the number of flagged days and $Y$ the sample length in **years**.

### C. Jump intensity $\lambda$ (estimated once)

$$\hat\lambda = \frac{n_{\text{jumps}}}{Y}$$

Examples: $\lambda=0.5$ ≈ one jump every two years; $\lambda=2$ ≈ two jumps per year.

### D. Jump size parameters (estimated once)

On flagged jump days, treat $J_t \approx r_t$ (or $r_t$ minus a typical diffusion day):

$$\hat\mu_J = \overline{J},\qquad \hat\sigma_J = \mathrm{std}(J)$$

### E. Compensation $\kappa$ (derived, not estimated separately)

$$\kappa = e^{\hat\mu_J + \hat\sigma_J^2/2} - 1$$

### F. More advanced (optional later)

Maximum likelihood for Merton (mixture of Poisson number of normals each day), or calibrate to option prices. For this project, threshold + moments is enough to understand the pipeline.

### Data pointers

- Returns: `../data/equity/log_returns_by_regime.csv`
- Compare crisis vs normal: expect larger $\hat\lambda$ and more negative $\hat\mu_J$ in crisis.


## 4. Constant vs path-updating parameters

### Calibrated once (fixed for the whole Monte Carlo run)

| Parameter | Role | Updates during a path? |
|-----------|------|------------------------|
| $\mu$ | drift | **No** |
| $\sigma$ | diffusion volatility | **No** |
| $\lambda$ | jump intensity | **No** |
| $\mu_J,\sigma_J$ | jump-size law | **No** |
| $\kappa$ | compensation (from $\mu_J,\sigma_J$) | **No** |
| $S_0,T,\Delta t$ | design | **No** |

### Evolve / redraw along each path

| Quantity | Role | Updates during a path? |
|----------|------|------------------------|
| $S_t$ | price | **Yes** — forms the path |
| $Z_t$ | diffusion shock | **Yes** — new $N(0,1)$ each step |
| $N_{\Delta t}$ | # jumps this step | **Yes** — new $\mathrm{Poisson}(\lambda\Delta t)$ each step |
| $J_i$ | jump sizes | **Yes** — drawn when $N_{\Delta t}>0$ |

Important: $\lambda$ is constant, but the **realized** number of jumps on a given day is random and can be 0 most days.

### One simulation step

1. Keep $(\mu,\sigma,\lambda,\mu_J,\sigma_J,\kappa)$ fixed.
2. Draw $Z\sim N(0,1)$ and $N\sim\mathrm{Poisson}(\lambda\Delta t)$.
3. If $N>0$, draw $J_1,\ldots,J_N\sim N(\mu_J,\sigma_J^2)$; else jump sum $=0$.
4. Update $S$ with the exponential formula above.


## 5. Interactive playground

Raise $\lambda$ or $\sigma_J$ to see more / larger jumps (kinks in paths, fatter return tails). Diffusion sliders behave like GBM.


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

%matplotlib inline

def simulate_merton(mu, sigma, lam, mu_j, sigma_j, S0, T, n_steps, n_paths, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    kappa = np.exp(mu_j + 0.5 * sigma_j**2) - 1.0
    z = rng.standard_normal((n_paths, n_steps))
    n_jumps = rng.poisson(lam * dt, size=(n_paths, n_steps))
    jump_sizes = np.zeros_like(z)
    mask = n_jumps > 0
    # compound Poisson: sum of n_jumps normal jumps
    jump_sizes[mask] = (
        n_jumps[mask] * mu_j
        + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(mask.sum())
    )
    increments = (mu - 0.5 * sigma**2 - lam * kappa) * dt + sigma * np.sqrt(dt) * z + jump_sizes
    log_paths = np.cumsum(increments, axis=1)
    paths = S0 * np.exp(np.hstack([np.zeros((n_paths, 1)), log_paths]))
    t = np.linspace(0, T, n_steps + 1)
    return t, paths, increments

def plot_merton(
    mu=0.08, sigma=0.18, lam=0.5, mu_j=-0.05, sigma_j=0.10,
    S0=100.0, T=1.0, n_steps=500, n_paths=1000,
):
    t, paths, increments = simulate_merton(
        mu, sigma, lam, mu_j, sigma_j, S0, T, n_steps, n_paths
    )

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(t, paths.T, alpha=0.35, lw=0.9)
    axes[0].plot(t, paths.mean(axis=0), color="black", lw=2, label="mean path")
    axes[0].set_title("Merton Monte Carlo paths")
    axes[0].set_xlabel("years")
    axes[0].set_ylabel("price")
    axes[0].legend(loc="upper left")

    axes[1].hist(increments.ravel(), bins=80, density=True, alpha=0.75, color="darkorange")
    axes[1].set_title("Step log-return distribution (with jumps)")
    axes[1].set_xlabel("log return")
    axes[1].set_ylabel("density")

    fig.suptitle(
        f"μ={mu:.2f}, σ={sigma:.2f}, λ={lam:.2f}, μJ={mu_j:.2f}, σJ={sigma_j:.2f}",
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()

interact(
    plot_merton,
    mu=FloatSlider(value=0.08, min=-0.20, max=0.40, step=0.01, description="μ"),
    sigma=FloatSlider(value=0.18, min=0.01, max=0.60, step=0.01, description="σ"),
    lam=FloatSlider(value=0.5, min=0.0, max=5.0, step=0.1, description="λ jumps/y"),
    mu_j=FloatSlider(value=-0.05, min=-0.40, max=0.20, step=0.01, description="μ_J"),
    sigma_j=FloatSlider(value=0.10, min=0.01, max=0.50, step=0.01, description="σ_J"),
    S0=FloatSlider(value=100.0, min=10.0, max=500.0, step=5.0, description="S0"),
    T=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="T"),
    n_steps=IntSlider(value=500, min=50, max=2000, step=10, description="steps"),
    n_paths=IntSlider(value=1000, min=5, max=2000, step=5, description="paths"),
);

interactive(children=(FloatSlider(value=0.08, description='μ', max=0.4, min=-0.2, step=0.01), FloatSlider(valu…